# 03 - Lakeflow Connect: Ingesta con Auto Loader

**Demo instructor:** [Tour oficial](https://www.databricks.com/resources/demos/tours/platform/discover-databricks-lakeflow-connect-demo)

## Objetivo
Demostrar la ingesta incremental de archivos usando **Auto Loader** (`cloudFiles`), el patrón equivalente a Lakeflow Connect para fuentes basadas en archivos.

## Flujo del notebook
1. **Cargar variables** — importa catálogo, esquemas y rutas del volumen desde `00_variables`
2. **Leer stream** — usa Auto Loader para detectar automáticamente archivos CSV nuevos en el volumen de transacciones (esquema inferido y almacenado en `_schemas/`)
3. **Escribir a Delta** — persiste el stream como tabla Delta en la capa bronze con checkpoint para procesamiento exactamente-una-vez

## Duración: ~10 minutos (demo instructor)


In [0]:
%run "./00 - Setup/00_variables"


In [0]:
# Auto Loader (cloudFiles): detecta archivos nuevos en el volumen automáticamente
# - cloudFiles.format: formato de los archivos fuente (CSV)
# - header: primera fila como nombres de columna
# - schemaLocation: guarda el esquema inferido para evolución automática
df = (spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").option("header","true").option("cloudFiles.schemaLocation", f"{vol_path}/_schemas/transacciones").load(f"{vol_path}/transacciones/"))
display(df.limit(5))


In [0]:
# Escribe el stream a una tabla Delta en la capa bronze
# - checkpointLocation: garantiza procesamiento exactamente-una-vez (exactly-once)
# - trigger(availableNow=True): procesa todos los archivos disponibles y termina
# - .table(): destino como tabla gestionada en Unity Catalog
(df.writeStream.format("delta").option("checkpointLocation", f"{vol_path}/_checkpoints/demo").trigger(availableNow=True).table(f"{catalog_name}.{schema_bronze}.transacciones_connect_demo").awaitTermination())
display(spark.sql(f"SELECT COUNT(*) as total FROM {catalog_name}.{schema_bronze}.transacciones_connect_demo"))
